# Nettoyage des données

Chaque décision est justifiée et chiffrée (journal de nettoyage). Le fichier brut `data/raw/hotel_bookings.csv` n'est jamais modifié : le résultat nettoyé est enregistré dans `data/processed/`.

In [1]:
import pandas as pd

hotels = pd.read_csv(
    "../data/raw/hotel_bookings.csv",
    parse_dates=["reservation_status_date"],
    dtype={"agent": "Int64", "company": "Int64"},
)
nb_lignes_depart = len(hotels)
nb_lignes_depart

119390

In [2]:
numero_mois = {mois: numero for numero, mois in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"],
    start=1,
)}

hotels["arrival_date"] = pd.to_datetime(dict(
    year=hotels["arrival_date_year"],
    month=hotels["arrival_date_month"].map(numero_mois),
    day=hotels["arrival_date_day_of_month"],
))
hotels[["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month", "arrival_date"]].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date
0,2015,July,1,2015-07-01
1,2015,July,1,2015-07-01
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01


### Décision 1 : date d'arrivée

| Quoi | Pourquoi | Lignes |
|---|---|---|
| colonne `arrival_date` créée | l'arrivée était éclatée en 3 colonnes (année, mois en texte, jour) : impossible de faire une analyse par période | 119390 |

In [3]:
nb_doublons = hotels.duplicated().sum()
part_doublons = round(nb_doublons / len(hotels) * 100, 1)
print(f"Doublons : {nb_doublons} lignes ({part_doublons} %)")

Doublons : 31994 lignes (26.8 %)


In [4]:
copies = hotels[hotels.duplicated()]
(copies["market_segment"].value_counts(normalize=True) * 100).round(0)

market_segment
Groups           46.0
Offline TA/TO    32.0
Online TA        15.0
Corporate         3.0
Direct            3.0
Complementary     0.0
Aviation          0.0
Name: proportion, dtype: float64

In [5]:
taux_avec_doublons = hotels["is_canceled"].mean() * 100
taux_sans_doublons = hotels.drop_duplicates()["is_canceled"].mean() * 100
print(f"Taux d'annulation avec doublons : {taux_avec_doublons:.1f} %")
print(f"Taux d'annulation sans doublons : {taux_sans_doublons:.1f} %")

Taux d'annulation avec doublons : 37.0 %
Taux d'annulation sans doublons : 27.5 %


In [6]:
hotels = hotels.drop_duplicates().reset_index(drop=True)
print(f"Lignes avant : {nb_lignes_depart} / après : {len(hotels)}")

Lignes avant : 119390 / après : 87396


### Décision 2 : doublons

| Quoi | Pourquoi | Lignes |
|---|---|---|
| doublons supprimés | aucun identifiant de réservation : une ligne identique sur toutes les colonnes est très probablement la même réservation enregistrée deux fois. Les garder gonflerait le taux d'annulation (37.0 % contre 27.5 % sans eux) | 31994 |

**Limite :** 46 % de ces doublons sont des réservations de groupes, qui pourraient être de vraies réservations identiques (plusieurs chambres réservées en même temps). Le choix est donc défendable mais pas certain.